In [0]:
# =============================================================================
# F1-Pulse | Silver Layer — Cleaning & Enrichment
# Notebook: 02_Silver_Transformation
# Author:   Jafar891
# Updated:  2026
#
# Reads Bronze Delta tables, applies schema enforcement, type casting,
# filtering, and enrichment joins. Writes clean data to the Silver layer.
# Idempotent — safe to re-run.
# =============================================================================

import sys, importlib
import logging

sys.path.append("/Workspace/Users/jafar.gahramanov@gmail.com/F1-Pulse")

# Reload modules so Databricks picks up any edits without a cluster restart
import modules.silver_helpers as silver_helpers
import modules.silver_transforms as silver_transforms
importlib.reload(silver_helpers)
importlib.reload(silver_transforms)

from modules.silver_helpers import read_bronze, write_silver
from modules.silver_transforms import transform_sessions, transform_laps

from config.config import (
    YEAR, CATALOG, BRONZE, SILVER, MIN_LAP_DURATION_S
)

# ---------------------------------------------------------------------------
# Logging
# ---------------------------------------------------------------------------
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)
log = logging.getLogger("f1_pulse.silver")


# ---------------------------------------------------------------------------
# Orchestration
# ---------------------------------------------------------------------------

def run_silver() -> None:

    log.info("=" * 60)
    log.info(f"F1-Pulse Silver Transformation — {YEAR}")
    log.info("=" * 60)

    results = {"success": [], "failed": []}

    # ------------------------------------------------------------------
    # Task 1: Sessions
    # ------------------------------------------------------------------
    log.info("\n[1/2] Transforming sessions …")
    try:
        bronze_sessions = read_bronze(spark, CATALOG, BRONZE, f"raw_sessions_{YEAR}")
        silver_sessions = transform_sessions(bronze_sessions, pipeline_year=YEAR)
        write_silver(silver_sessions, CATALOG, SILVER, "cleaned_sessions")
        results["success"].append("cleaned_sessions")
    except Exception as e:
        log.error(f"  ❌ Sessions transformation failed: {e}")
        results["failed"].append("cleaned_sessions")

    # ------------------------------------------------------------------
    # Task 2: Enriched laps
    # ------------------------------------------------------------------
    log.info("\n[2/2] Enriching laps …")
    try:
        bronze_laps    = read_bronze(spark, CATALOG, BRONZE, f"raw_laps_{YEAR}")
        bronze_drivers = read_bronze(spark, CATALOG, BRONZE, f"raw_drivers_{YEAR}")
        silver_laps    = transform_laps(bronze_laps, bronze_drivers, MIN_LAP_DURATION_S)
        write_silver(silver_laps, CATALOG, SILVER, "enriched_laps")
        results["success"].append("enriched_laps")
    except Exception as e:
        log.error(f"  ❌ Laps enrichment failed: {e}")
        results["failed"].append("enriched_laps")

    # ------------------------------------------------------------------
    # Summary
    # ------------------------------------------------------------------
    log.info("\n" + "=" * 60)
    log.info("SILVER SUMMARY")
    log.info("=" * 60)
    for t in results["success"]:
        log.info(f"  ✅ {CATALOG}.{SILVER}.{t}")
    for t in results["failed"]:
        log.warning(f"  ❌ {t} — check logs above")
    log.info("=" * 60)

    if results["failed"]:
        raise RuntimeError(
            f"Silver layer completed with errors: {results['failed']}"
        )


# ---------------------------------------------------------------------------
# Entry point
# ---------------------------------------------------------------------------

run_silver()
